# Evaluate models · scoring, naive baseline and models 1–4

This notebook walks through how every model is scored and compared. It mirrors three scripts: `src/scoring.py` (metrics), `src/models.py` (the baseline and the four Bayesian models) and `src/evaluate_models.py` (the loop over folds). The functions below are copied from those files.

**To keep it quick, this notebook runs one fold with small fits.** The full comparison (all folds, all scenarios, full-size fits) is run from the terminal:

```
python src/evaluate_models.py --scenario clusters
```

and is summarized at the end of this notebook if its results exist.

> All data is simulated. `true_` columns are never used here.

## Setup

In [1]:
from pathlib import Path
import json
import logging
import sys
import time
import warnings

import numpy as np
import pandas as pd
import pymc as pm
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore")
logging.getLogger("pymc").setLevel(logging.ERROR)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from build_folds import PROCESSED, load_fold

RESULTS = ROOT / "results"
SCENARIO = "clusters"
FOLD = "fold_1"


In [2]:
SEED = 20261102


QUICK = dict(n_train=10_000, draws=500, tune=1000, chains=2, n_pred_draws=100)


PRECISION_AT_K = 10


MODELS = ("naive", "model_1", "model_2", "model_3", "model_4")


BAYES = {
    "model_1": dict(clusters=False, suppliers=None),
    "model_2": dict(clusters=True, suppliers=None),
    "model_3": dict(clusters=False, suppliers="shared"),
    "model_4": dict(clusters=True, suppliers="by_cluster"),
}


HEADS = ("late", "mu", "sigma")


SUPPLIER_EFFECT_HEADS = ("late", "mu")


NAIVE_MIN_SHIPMENTS = 20


PRIORS = {   # (intercept mean, intercept sd, coefficient sd) on the standardized scale
    "late": (-2.0, 1.5, 0.5),           # ~12% late at the intercept
    "mu": (1.3, 1.0, 0.5),              # median late delay ~3.7 h
    "sigma": (np.log(0.7), 0.5, 0.3),
}


MAX_PRED_DELAY_HOURS = 336.0

cfg = QUICK   # small fits so the notebook runs in a few minutes
cfg

{'n_train': 10000,
 'draws': 500,
 'tune': 1000,
 'chains': 2,
 'n_pred_draws': 100}

## Step 1 · How forecasts are scored

Every model, including the naive baseline, returns the same thing: a **matrix of predictive samples**, one row per test shipment and one column per draw. Each draw is a delay in hours, with 0 meaning on time. All metrics are computed from these samples, so every model goes through exactly the same scoring code.

### CRPS, the primary metric

The **continuous ranked probability score** compares the whole predicted distribution to what actually happened. Lower is better, and it's in hours, like MAE. A forecast that says "probably on time, small chance of a long delay" is rewarded when it's right about *both* parts. A point forecast can't express that.

From samples, CRPS = E|X − y| − ½·E|X − X′|. The first term is how far the forecast is from the truth. The second subtracts the forecast's own spread, so a model can't score well just by being vague.

In [3]:
def crps_from_samples(samples: np.ndarray, y: np.ndarray) -> np.ndarray:
    """CRPS per observation from an (n, S) matrix of predictive draws.

    CRPS = E|X - y| - 0.5 E|X - X'|. The second term uses the sorted-sample
    identity E|X - X'| = (2 / S^2) * sum_i (2i - S - 1) x_(i), which is O(S log S).
    """
    s = np.sort(samples, axis=1)
    n_draws = s.shape[1]
    term1 = np.abs(s - y[:, None]).mean(axis=1)
    weights = 2 * np.arange(1, n_draws + 1) - n_draws - 1
    term2 = (s * weights).sum(axis=1) / n_draws**2
    return term1 - term2

**Sanity check:** for a standard normal forecast there's an exact formula. The sample version should match it closely.

In [4]:
from math import erf, exp, pi, sqrt
y, draws = 0.3, np.random.default_rng(0).normal(0, 1, (1, 20_000))
exact = y * erf(y / sqrt(2)) + 2 * exp(-y * y / 2) / sqrt(2 * pi) - 1 / sqrt(pi)
print(f"sample CRPS {crps_from_samples(draws, np.array([y]))[0]:.4f}  vs exact {exact:.4f}")

sample CRPS 0.2699  vs exact 0.2693


### The full metric set

**Per shipment (each truck counted once):**

| Metric | Question it answers |
|---|---|
| `crps` | How good is the whole forecast distribution? (primary) |
| `mae_median`, `rmse_mean` | How far off is a single-number forecast? |
| `bias_mean` | Does the model under-predict lateness? (negative = yes, the dangerous direction) |
| `brier_late`, `log_loss_late` | How accurate is P(late)? |
| `coverage_90`, `width_90` | Do 90% intervals contain the truth 90% of the time, and how wide are they? |
| `pinball_q90` | How well is the bad tail predicted? |
| `crps_cold_start` | CRPS for new suppliers only, where hierarchical models should help most |

**Per delivery row (part × plant), for the 72-hour shortage step:**

| Metric | Question it answers |
|---|---|
| `brier_shortage` | How accurate is P(delay > safety stock coverage)? |
| `pr_auc_shortage` | How well are real shortages ranked above non-shortages? |
| `precision_at_10` | Of the top 10 risks flagged each day, how many were real shortages? |

In [5]:
def pinball(samples: np.ndarray, y: np.ndarray, q: float) -> np.ndarray:
    pred = np.quantile(samples, q, axis=1)
    diff = y - pred
    return np.maximum(q * diff, (q - 1) * diff)


def delay_metrics(samples: np.ndarray, y: np.ndarray, late: np.ndarray) -> dict:
    """Shipment-level metrics for one model on one test window."""
    p_late = (samples > 0).mean(axis=1)
    p_clip = np.clip(p_late, 1e-4, 1 - 1e-4)
    lo, hi = np.quantile(samples, [0.05, 0.95], axis=1)
    mean = samples.mean(axis=1)
    return {
        "crps": crps_from_samples(samples, y).mean(),
        "mae_median": np.abs(np.median(samples, axis=1) - y).mean(),
        "rmse_mean": np.sqrt(((mean - y) ** 2).mean()),
        "bias_mean": (mean - y).mean(),                      # negative = under-predicts lateness
        "brier_late": ((p_late - late) ** 2).mean(),
        "log_loss_late": -(late * np.log(p_clip) + (1 - late) * np.log(1 - p_clip)).mean(),
        "coverage_90": ((y >= lo) & (y <= hi)).mean(),
        "width_90": (hi - lo).mean(),
        "pinball_q90": pinball(samples, y, 0.90).mean(),
    }


def shortage_metrics(row_samples: np.ndarray, coverage_hours: np.ndarray,
                     shortage: np.ndarray, day: np.ndarray) -> dict:
    """Row-level shortage metrics: P(delay > safety stock coverage)."""
    p_short = (row_samples > coverage_hours[:, None]).mean(axis=1)
    top = (pd.DataFrame({"day": day, "p": p_short, "y": shortage})
           .sort_values("p", ascending=False).groupby("day").head(PRECISION_AT_K))
    return {
        "brier_shortage": ((p_short - shortage) ** 2).mean(),
        "pr_auc_shortage": average_precision_score(shortage, p_short) if shortage.any() else np.nan,
        f"precision_at_{PRECISION_AT_K}": top["y"].mean(),
    }


def score(samples: np.ndarray, test_ships: pd.DataFrame, test_rows: pd.DataFrame) -> dict:
    """All metrics for one model on one fold's test window.

    test_ships: one row per shipment, aligned with `samples`
    test_rows:  one row per delivery row, with a `ship_idx` column pointing
                into test_ships, plus safety_coverage_hours and shortage_flag
    """
    y = test_ships["delay_hours"].to_numpy(float)
    late = (y > 0).astype(float)
    out = delay_metrics(samples, y, late)

    cold = test_ships["is_cold"].to_numpy(bool)
    out["crps_cold_start"] = crps_from_samples(samples[cold], y[cold]).mean() if cold.any() else np.nan
    out["n_cold_start"] = int(cold.sum())

    rows = test_rows.dropna(subset=["shortage_flag", "safety_coverage_hours"])
    out.update(shortage_metrics(samples[rows["ship_idx"].to_numpy()],
                                rows["safety_coverage_hours"].to_numpy(float),
                                rows["shortage_flag"].to_numpy(bool).astype(float),
                                rows["production_date"].to_numpy()))
    out["n_test_shipments"] = len(y)
    return out

## Step 2 · Prepare a fold

`load_fold` (from the folds step) returns imputed train and test rows with this fold's cluster labels. These are then:

- **Collapsed to shipments,** because parts on one truck share a single delay.
- **Subsampled for the "is it late?" part** of the Bayesian models, to keep MCMC fast. Every model in a fold uses the *same* subsample, so comparisons are fair. The "how late?" part uses every late training shipment.
- **Filtered for scoring:** test shipments whose receipt was never scanned have no true delay, so they aren't scored. Each test row keeps a `ship_idx` pointing to its shipment's forecast.

In [6]:
def to_shipments(rows: pd.DataFrame) -> pd.DataFrame:
    """One row per truck (parts on a truck share one delay)."""
    ships = rows.drop_duplicates("shipment_id").reset_index(drop=True)
    ships["is_cold"] = ships["sup_is_cold_start"].astype(bool) | (ships["cluster"] == "NEW")
    return ships


def fold_seed(*parts: str) -> int:
    return int(np.random.SeedSequence([SEED, *[sum(map(ord, p)) for p in parts]]).generate_state(1)[0])


def prepare_fold(scenario: str, fold_id: str):
    train_rows, test_rows = load_fold(scenario, fold_id)
    train_ships = to_shipments(train_rows)

    # Score only shipments whose delay is known (unscanned receipts have no truth)
    test_ships = to_shipments(test_rows).dropna(subset=["delay_hours"]).reset_index(drop=True)
    ship_idx = pd.Series(np.arange(len(test_ships)), index=test_ships["shipment_id"])
    test_rows = test_rows[test_rows["shipment_id"].isin(ship_idx.index)].copy()
    test_rows["ship_idx"] = test_rows["shipment_id"].map(ship_idx).to_numpy()
    return train_ships, test_ships, test_rows

In [7]:
train_ships, test_ships, test_rows = prepare_fold(SCENARIO, FOLD)
subsample_seed = fold_seed(SCENARIO, FOLD) % 2**32
print(f"train {len(train_ships):,} shipments | test {len(test_ships):,} shipments, {len(test_rows):,} rows")
print(f"test late rate {(test_ships['delay_hours'] > 0).mean():.3f} | "
      f"shortage rate {test_rows['shortage_flag'].astype(float).mean():.3f} | "
      f"cold-start shipments {test_ships['is_cold'].sum():,}")

train 155,548 shipments | test 18,991 shipments, 32,947 rows
test late rate 0.112 | shortage rate 0.050 | cold-start shipments 254


## Step 3 · Naive baseline

The floor every model must beat, with no fitting at all. Each test shipment's forecast is the **empirical distribution of its supplier's delays in the training window**: if 8% of a supplier's past trucks were late, about 8% of the draws will be past delays and the rest zeros. Suppliers with fewer than 20 training shipments use everyone's delays pooled.

It's deliberately static: it doesn't react to recent changes. In the drift scenario it should do worst.

In [8]:
def naive_samples(train_ships: pd.DataFrame, test_ships: pd.DataFrame,
                  n_draws: int, rng: np.random.Generator) -> np.ndarray:
    known = train_ships.dropna(subset=["delay_hours"])
    pooled = known["delay_hours"].to_numpy()
    by_supplier = {s: g.to_numpy() for s, g in known.groupby("supplier_code")["delay_hours"]
                   if len(g) >= NAIVE_MIN_SHIPMENTS}
    out = np.empty((len(test_ships), n_draws))
    for supplier, idx in test_ships.groupby("supplier_code").indices.items():
        source = by_supplier.get(supplier, pooled)
        out[idx] = rng.choice(source, size=(len(idx), n_draws))
    return out

In [9]:
results = {}
rng = np.random.default_rng(1)
samples = naive_samples(train_ships, test_ships, cfg["n_pred_draws"], rng)
results["naive"] = score(samples, test_ships, test_rows)
pd.Series(results["naive"]).round(4).to_frame("naive")

,naive
crps,0.7083
mae_median,0.8262
rmse_mean,3.7468
bias_mean,0.0263
brier_late,0.0883
log_loss_late,0.3042
coverage_90,0.9585
width_90,4.1089
pinball_q90,0.5102
crps_cold_start,1.3151


## Step 4 · The Bayesian hurdle models

### Predictors

Each head gets only the predictors that matter to it:

| Head | Predictors |
|---|---|
| late | late rates, recent change in late rate, shipment count, days since last late, cold-start flag, region 7-day late rate, winter, hour, trucks per day, region |
| mu (how late) | typical late delay, its recent change, cold-start flag, region 7-day late rate, winter, region |
| sigma (how variable) | delay spread, its recent change, cold-start flag |

Three choices keep the posterior easy to sample:

- **Transform to a linear scale:** late rates on the logit scale, delays on the log scale.
- **Enter overlapping windows as differences:** "30-day rate minus 90-day rate" instead of both rates. The 30-day, 90-day and all-history rates are nearly identical for a stable supplier, and near-duplicate predictors made the first version of model 1 take 12 minutes and mix poorly.
- **Standardize using training means and SDs only.**

**For the cluster models (2 and 4), predictors are also centred on their cluster's mean.** Clusters are built from these same history features, so the cluster effects and the feature coefficients would otherwise explain the same thing, and the sampler couldn't separate them (model 4 first reached r-hat 1.14). After centring, cluster effects capture differences *between* clusters and coefficients capture differences *within* a cluster.

In [10]:
def _logit(p):
    p = np.clip(p, 0.005, 0.995)
    return np.log(p / (1 - p))


def head_predictors(df: pd.DataFrame, head: str) -> pd.DataFrame:
    """Transformed predictors for one head. Rates on the logit scale, delays on
    the log scale; overlapping windows entered as differences."""
    late_all, late_90, late_30 = (_logit(df[f"sup_late_rate_{w}"]) for w in ("all", "90d", "30d"))
    delay_all = np.log(df["sup_mean_late_delay_all"].clip(lower=0.05))
    delay_90 = np.log(df["sup_mean_late_delay_90d"].clip(lower=0.05))
    common = {
        "cold_start": df["sup_is_cold_start"].astype(float),
        "region_late_7d": df["region_late_rate_7d"],
        "is_winter": df["is_winter"].astype(float),
    }
    if head == "late":
        cols = {
            "logit_late_all": late_all,
            "late_90d_vs_all": late_90 - late_all,
            "late_30d_vs_90d": late_30 - late_90,
            "log_n_ship": np.log1p(df["sup_n_ship_all"]),
            "log_days_since_late": np.log1p(df["sup_days_since_late"].clip(lower=0)),
            "planned_hour": df["planned_hour"],
            "trips_per_day": df["trips_per_day"],
            **common,
        }
    elif head == "mu":
        cols = {"log_late_delay_all": delay_all, "late_delay_90d_vs_all": delay_90 - delay_all, **common}
    else:
        cols = {
            "log_delay_sd_all": df["sup_log_delay_sd_all"],
            "delay_sd_90d_vs_all": df["sup_log_delay_sd_90d"] - df["sup_log_delay_sd_all"],
            "cold_start": common["cold_start"],
        }
    x = pd.DataFrame(cols, index=df.index)
    if head in ("late", "mu"):
        regions = pd.get_dummies(df["region"], prefix="region", dtype=float)
        x = pd.concat([x, regions], axis=1)
    return x


class Design:
    """Per-head predictors, standardized with training means and SDs only.

    For the cluster models (2 and 4), `groups` holds each row's cluster and every
    predictor is also centred on its training cluster mean (a within/between
    split). Clusters are built from these same history features, so without
    centring the cluster effects and the feature coefficients explain the same
    thing and the sampler cannot separate them. After centring, cluster effects
    capture differences between clusters and coefficients capture differences
    within a cluster.
    """

    def fit(self, train: pd.DataFrame, head: str, groups: pd.Series | None = None) -> "Design":
        self.head = head
        x = head_predictors(train, head)
        region_cols = sorted(c for c in x.columns if c.startswith("region_"))
        x = x.drop(columns=region_cols[:1])            # reference region, avoids collinearity
        self.columns = list(x.columns)
        self.mean, self.sd = x.mean(), x.std().replace(0, 1)
        self.group_means = None
        if groups is not None:
            z = ((x - self.mean) / self.sd).fillna(0.0)
            self.group_means = z.groupby(groups.to_numpy()).mean()
        return self

    def transform(self, df: pd.DataFrame, groups: pd.Series | None = None) -> np.ndarray:
        x = head_predictors(df, self.head).reindex(columns=self.columns, fill_value=0.0)
        z = ((x - self.mean) / self.sd).fillna(0.0)
        if self.group_means is not None:
            z = z - self.group_means.reindex(groups.to_numpy()).fillna(0.0).to_numpy()
        return z.to_numpy(float)

### One structure, four variants

Each model is a **hurdle model** with three linear predictors (heads):

```
late_i         ~ Bernoulli(logit⁻¹(η_i))          will the truck be late?
log(delay_i)   ~ Normal(μ_i, σ_i)   if late        how late (log scale)

η_i     = a₀ + X_i·b + effects
μ_i     = c₀ + X_i·g + effects
log σ_i = d₀ + X_i·h + effects                    volatility can differ by supplier too
```

The four models differ only in the **effects** term, giving the 2×2 design:

| | No clusters | Clusters |
|---|---|---|
| **No supplier effects** | **Model 1:** suppliers differ only through history features | **Model 2:** + a fixed effect per cluster |
| **Supplier effects (hierarchical)** | **Model 3:** effect_s ~ Normal(0, τ) | **Model 4:** cluster effects + effect_s ~ Normal(0, τ_cluster) |

**Partial pooling (models 3 and 4):** each supplier gets its own adjustment, but adjustments are pulled toward zero by an amount the model learns (τ). A supplier with little data gets pulled in strongly. One with lots of data keeps its own estimate. Model 4 does the pulling *within each cluster*, and each cluster gets its own spread.

**New suppliers:** a supplier not seen in training gets a fresh effect drawn from the population (model 3) or from its cluster's population (model 4). That's how these models handle cold start.

**Supplier effects go on the late and mu heads only.** With about 10 late shipments per supplier, a separate volatility per supplier is barely identified and made the sampler fail.

**Two separate fits:** the "is it late?" part and the "how late?" part share no parameters, so they are fit as two PyMC models. This gives the same posterior as one joint model, but lets the severity part use every late shipment.

**Prediction cap:** predicted delays are capped at two weeks (336 h). Operationally, a truck that late is a cancellation, not a delay.

Supplier effects use a **non-centered** form (τ·z with z ~ Normal(0, 1)), which samples much more reliably when some suppliers have little data.

**Priors** are weakly informative on the standardized scale. The intercepts are centred on plausible values: about 12% of trucks late, and a median delay of about 3.7 hours when late.

In [11]:
class HurdleModel:
    """One of model_1..model_4, fit with PyMC and predicted in NumPy."""

    def __init__(self, name: str):
        self.name = name
        self.use_clusters = BAYES[name]["clusters"]
        self.supplier_effects = BAYES[name]["suppliers"]

    def _cluster_dummies(self, clusters: pd.Series) -> np.ndarray:
        return np.column_stack([(clusters == c).to_numpy(float) for c in self.cluster_levels[1:]]) \
            if len(self.cluster_levels) > 1 else np.zeros((len(clusters), 0))

    def _add_head(self, head: str, data: pd.DataFrame) -> "pm.TensorVariable":
        """Linear predictor for one head, inside the active PyMC model."""
        groups = data["cluster"] if self.use_clusters else None
        design = Design().fit(data, head, groups)
        self.designs[head] = design
        X = design.transform(data, groups)
        loc, scale, coef_sd = PRIORS[head]
        intercept = pm.Normal(f"{head}_intercept", loc, scale)
        beta = pm.Normal(f"{head}_beta", 0, coef_sd, shape=X.shape[1])
        lp = intercept + pm.math.dot(X, beta)

        if self.use_clusters and len(self.cluster_levels) > 1:
            # cluster shifts get the same prior scale as that head's coefficients, x2;
            # a wide prior on the sigma head let small clusters (e.g. NEW) blow up the tail
            alpha = pm.Normal(f"{head}_cluster", 0, 2 * coef_sd, shape=len(self.cluster_levels) - 1)
            lp = lp + pm.math.dot(self._cluster_dummies(data["cluster"]), alpha)

        if self.supplier_effects and head in SUPPLIER_EFFECT_HEADS:
            sup_idx = np.searchsorted(self.suppliers, data["supplier_code"].to_numpy())
            if self.supplier_effects == "shared":
                tau = pm.HalfNormal(f"{head}_tau", 0.5)
                spread = tau
            else:
                tau = pm.HalfNormal(f"{head}_tau", 0.5, shape=len(self.cluster_levels))
                spread = tau[self.sup_cluster]
            z = pm.Normal(f"{head}_z", 0, 1, shape=len(self.suppliers))         # non-centred
            effect = pm.Deterministic(f"{head}_effect", spread * z)
            lp = lp + effect[sup_idx]
        return lp

    def fit(self, train_ships: pd.DataFrame, n_occurrence: int, draws: int, tune: int,
            chains: int, seed: int, subsample_seed: int = 0, sampler: str = "nutpie"):
        data = train_ships.dropna(subset=["delay_hours"]).reset_index(drop=True)
        self.designs, self.idata = {}, {}

        # Shared indexing: suppliers, clusters (C1 = reference), each supplier's cluster
        self.cluster_levels = sorted(data["cluster"].unique(), key=lambda c: (c == "NEW", c))
        self.suppliers = np.array(sorted(data["supplier_code"].unique()))
        level = {c: i for i, c in enumerate(self.cluster_levels)}
        self.sup_cluster = (data.drop_duplicates("supplier_code").set_index("supplier_code")
                            .loc[self.suppliers, "cluster"].map(level).to_numpy())

        occurrence = data.sample(min(n_occurrence, len(data)), random_state=subsample_seed)
        severity = data[data["delay_hours"] > 0]
        kwargs = dict(draws=draws, tune=tune, chains=chains, random_seed=seed, progressbar=False)
        if sampler == "nutpie":
            kwargs["nuts_sampler"] = "nutpie"

        with pm.Model():
            lp = self._add_head("late", occurrence)
            pm.Bernoulli("late_obs", logit_p=lp, observed=(occurrence["delay_hours"] > 0).to_numpy(int))
            self.idata["occurrence"] = pm.sample(**kwargs)
        with pm.Model():
            mu = self._add_head("mu", severity)
            log_sigma = self._add_head("sigma", severity)
            pm.Normal("log_delay_obs", mu=mu, sigma=pm.math.exp(log_sigma),
                      observed=np.log(severity["delay_hours"].to_numpy(float)))
            self.idata["severity"] = pm.sample(**kwargs)
        return self

    # -- prediction -----------------------------------------------------------
    def predict_samples(self, test_ships: pd.DataFrame, n_draws: int, rng) -> np.ndarray:
        """(n_test, n_draws) predictive delays. Each column uses one posterior draw
        per part (occurrence, severity), shared by every parameter in that part."""
        n = len(test_ships)
        known = np.isin(test_ships["supplier_code"], self.suppliers)
        sup_idx = np.searchsorted(self.suppliers, test_ships["supplier_code"].to_numpy())
        sup_idx = np.where(known, np.minimum(sup_idx, len(self.suppliers) - 1), 0)
        level = {c: i for i, c in enumerate(self.cluster_levels)}
        cl_idx = test_ships["cluster"].map(level).fillna(level.get("NEW", 0)).to_numpy(int)

        draw_ids = {}
        for part, idata in self.idata.items():
            total = idata.posterior.sizes["chain"] * idata.posterior.sizes["draw"]
            draw_ids[part] = rng.choice(total, size=n_draws, replace=total < n_draws)

        def post(var: str) -> np.ndarray:
            part = "occurrence" if var.startswith("late") else "severity"
            p = self.idata[part].posterior[var].stack(sample=("chain", "draw")).transpose("sample", ...).to_numpy()
            return p[draw_ids[part]]

        lp = {}
        for head in HEADS:
            X = self.designs[head].transform(test_ships, test_ships["cluster"] if self.use_clusters else None)
            val = post(f"{head}_intercept")[None, :] + X @ post(f"{head}_beta").T            # (n, S)
            if self.use_clusters and len(self.cluster_levels) > 1:
                val = val + self._cluster_dummies(test_ships["cluster"]) @ post(f"{head}_cluster").T
            if self.supplier_effects and head in SUPPLIER_EFFECT_HEADS:
                effect, tau = post(f"{head}_effect"), post(f"{head}_tau")    # (S, n_sup), (S,) or (S, k)
                eff_known = effect[:, sup_idx].T
                if self.supplier_effects == "shared":
                    eff_new = tau[None, :] * rng.standard_normal((n, n_draws))
                else:
                    eff_new = tau[:, cl_idx].T * rng.standard_normal((n, n_draws))
                val = val + np.where(known[:, None], eff_known, eff_new)
            lp[head] = val

        late = rng.random((n, n_draws)) < 1 / (1 + np.exp(-lp["late"]))
        log_delay = lp["mu"] + np.exp(lp["sigma"]) * rng.standard_normal((n, n_draws))
        return np.where(late, np.exp(np.minimum(log_delay, np.log(MAX_PRED_DELAY_HOURS))), 0.0)

    def diagnostics(self) -> dict:
        import arviz as az
        out = {"max_rhat": 0.0, "min_ess_bulk": np.inf, "divergences": 0}
        for idata in self.idata.values():
            summ = az.summary(idata, kind="diagnostics")
            out["max_rhat"] = max(out["max_rhat"], float(summ["r_hat"].max()))
            out["min_ess_bulk"] = min(out["min_ess_bulk"], float(summ["ess_bulk"].min()))
            out["divergences"] += int(idata.sample_stats["diverging"].sum())
        return out

### Fit all four on this fold

With the small notebook settings, this takes a few minutes. Watch the convergence diagnostics:
- **`max_rhat`** should be below about 1.01.
- **`min_ess_bulk`** should be at least a few hundred.
- **`divergences`** should be 0.

If they're off with these small settings, the full-size run in the script is the one to trust.

In [12]:
def predict(model_name: str, train_ships, test_ships, cfg: dict, seed: int, sampler: str,
            subsample_seed: int = 0):
    rng = np.random.default_rng(seed)
    start = time.time()
    if model_name == "naive":
        samples, diag = naive_samples(train_ships, test_ships, cfg["n_pred_draws"], rng), {}
    else:
        model = HurdleModel(model_name).fit(train_ships, cfg["n_train"], cfg["draws"], cfg["tune"],
                                            cfg["chains"], seed % 2**32, subsample_seed, sampler)
        samples, diag = model.predict_samples(test_ships, cfg["n_pred_draws"], rng), model.diagnostics()
    return samples, diag, time.time() - start

In [13]:
fit_info = {}
for name in BAYES:
    samples, diag, seconds = predict(name, train_ships, test_ships, cfg, fold_seed(SCENARIO, FOLD, name),
                                     "nutpie", subsample_seed)
    results[name] = score(samples, test_ships, test_rows)
    fit_info[name] = {**diag, "seconds": round(seconds)}
    print(f"{name}: CRPS {results[name]['crps']:.4f} ({seconds:.0f}s)")
pd.DataFrame(fit_info).T

model_1: CRPS 0.7047 (51s)


model_2: CRPS 0.7074 (58s)


model_3: CRPS 0.7074 (69s)


model_4: CRPS 0.7068 (80s)


,max_rhat,min_ess_bulk,divergences,seconds
model_1,1.009345,170.916774,0.0,51.0
model_2,1.024498,263.961265,0.0,58.0
model_3,1.026807,97.477772,0.0,69.0
model_4,1.052088,56.427936,0.0,80.0


## Step 5 · Compare on this fold

Lower is better for every metric except `coverage_90` (target 0.90), `pr_auc_shortage` and `precision_at_10`.

In [14]:
cols = ["crps", "crps_cold_start", "mae_median", "bias_mean", "brier_late",
        "coverage_90", "width_90", "pinball_q90", "brier_shortage", "pr_auc_shortage", "precision_at_10"]
pd.DataFrame(results).T[cols].round(4)

,crps,crps_cold_start,mae_median,bias_mean,brier_late,coverage_90,width_90,pinball_q90,brier_shortage,pr_auc_shortage,precision_at_10
naive,0.7083,1.3151,0.8262,0.0263,0.0883,0.9585,4.1089,0.5102,0.0402,0.2716,0.4056
model_1,0.7047,1.2416,0.8247,0.0259,0.0882,0.9585,4.1000,0.5041,0.0401,0.2672,0.3944
model_2,0.7074,1.3535,0.8250,-0.0246,0.0891,0.9567,3.8408,0.5100,0.0403,0.2603,0.4333
model_3,0.7074,1.2294,0.8247,-0.0102,0.0885,0.9576,3.9120,0.5100,0.0404,0.2552,0.3389
model_4,0.7068,1.3747,0.8244,-0.0362,0.0891,0.9561,3.8109,0.5057,0.0402,0.2525,0.2889


One fold with small fits is a smoke test, not a result. Differences between models here can easily be noise. The real comparison needs every fold, and it uses the **paired** difference against the naive baseline:

- `crps_vs_ref`: mean CRPS difference across folds (negative = better than naive)
- `crps_vs_ref_sd`: how much that difference varies from fold to fold
- `folds_better`: in how many of the 4 folds the model beat naive. A gain that flips sign across folds isn't a win.

In [15]:
def summarize(metrics: pd.DataFrame, reference: str = "naive") -> pd.DataFrame:
    """Mean of each metric across folds, plus the paired CRPS difference vs a
    reference model (negative = better than the reference) and how many folds
    the model beat it in. A gain that flips sign across folds is not a win."""
    mean = metrics.groupby("model").mean(numeric_only=True)
    ref = metrics[metrics["model"] == reference].set_index("fold_id")["crps"]
    diffs = metrics.assign(crps_diff=metrics["crps"] - metrics["fold_id"].map(ref))
    paired = diffs.groupby("model")["crps_diff"].agg(
        crps_vs_ref="mean", crps_vs_ref_sd="std", folds_better=lambda d: int((d < 0).sum()))
    return mean.join(paired)

## Step 6 · The full run

`run` repeats Steps 2–5 for every fold and model. It saves `metrics.csv` after each model, so a long run can be stopped and resumed, and it writes per-shipment and per-row predictions for the dashboard. Run it from the terminal:

```
python src/evaluate_models.py --quick                      # smoke test
python src/evaluate_models.py --scenario clusters          # 4 CV folds x 5 models
python src/evaluate_models.py                              # all three scenarios
python src/evaluate_models.py --scenario clusters --holdout # final report: run ONCE, at the end
```

**Don't touch the holdout until the model choice is settled on the CV folds.** Running it more than once and keeping the best result is the overfitting this design exists to prevent.

In [16]:
def save_predictions(samples, test_ships, test_rows, out: Path, fold_id: str, model_name: str):
    out.mkdir(parents=True, exist_ok=True)
    q05, q50, q95 = np.quantile(samples, [0.05, 0.5, 0.95], axis=1)
    pd.DataFrame({
        "shipment_id": test_ships["shipment_id"], "supplier_code": test_ships["supplier_code"],
        "planned_delivery_datetime": test_ships["planned_delivery_datetime"],
        "delay_hours": test_ships["delay_hours"], "p_late": (samples > 0).mean(axis=1),
        "pred_mean": samples.mean(axis=1), "pred_q05": q05, "pred_median": q50, "pred_q95": q95,
    }).to_parquet(out / f"{fold_id}_{model_name}_shipments.parquet", index=False)

    rows = test_rows.dropna(subset=["safety_coverage_hours"])
    row_samples = samples[rows["ship_idx"].to_numpy()]
    pd.DataFrame({
        "delivery_id": rows["delivery_id"], "part_code": rows["part_code"], "plant": rows["plant"],
        "production_date": rows["production_date"],
        "p_shortage": (row_samples > rows["safety_coverage_hours"].to_numpy()[:, None]).mean(axis=1),
        "shortage_flag": rows["shortage_flag"],
    }).to_parquet(out / f"{fold_id}_{model_name}_rows.parquet", index=False)


def run(scenario: str, fold_ids: list[str], models: list[str], cfg: dict,
        sampler: str = "nutpie", overwrite: bool = False, tag: str = "") -> pd.DataFrame:
    out = RESULTS / f"{scenario}{tag}"
    out.mkdir(parents=True, exist_ok=True)
    metrics_path = out / "metrics.csv"
    metrics = pd.read_csv(metrics_path) if metrics_path.exists() and not overwrite else pd.DataFrame()
    done = set(zip(metrics.get("fold_id", []), metrics.get("model", [])))

    for fold_id in fold_ids:
        todo = [m for m in models if (fold_id, m) not in done]
        if not todo:
            continue
        train_ships, test_ships, test_rows = prepare_fold(scenario, fold_id)
        print(f"[{scenario} | {fold_id}] train {len(train_ships):,} shipments, "
              f"test {len(test_ships):,} shipments")
        subsample_seed = fold_seed(scenario, fold_id) % 2**32     # same subsample for every model
        for model_name in todo:
            samples, diag, seconds = predict(model_name, train_ships, test_ships, cfg,
                                             fold_seed(scenario, fold_id, model_name), sampler,
                                             subsample_seed)
            row = {"scenario": scenario, "fold_id": fold_id, "model": model_name,
                   **score(samples, test_ships, test_rows), "seconds": round(seconds, 1), **diag}
            metrics = pd.concat([metrics[~((metrics.get("fold_id") == fold_id) & (metrics.get("model") == model_name))]
                                 if len(metrics) else metrics, pd.DataFrame([row])], ignore_index=True)
            metrics.to_csv(metrics_path, index=False)          # save after every model (resumable)
            save_predictions(samples, test_ships, test_rows, out / "predictions", fold_id, model_name)
            flag = "" if model_name == "naive" or (diag.get("max_rhat", 1) < 1.01 and diag.get("divergences", 0) == 0) \
                else "  <-- check convergence"
            print(f"    {model_name:8s} CRPS {row['crps']:.4f}  brier_late {row['brier_late']:.4f}  "
                  f"cov90 {row['coverage_90']:.3f}  {seconds:5.0f}s{flag}")

    cv = metrics[metrics["fold_id"] != "holdout"]
    if len(cv) and "naive" in set(cv["model"]):
        summary = summarize(cv)
        summary.to_csv(out / "summary.csv")
        cols = ["crps", "crps_vs_ref", "crps_vs_ref_sd", "folds_better", "crps_cold_start",
                "brier_late", "coverage_90", "brier_shortage", "precision_at_10"]
        print(f"\n[{scenario}] mean over CV folds (crps_vs_ref < 0 = better than naive):")
        print(summary[[c for c in cols if c in summary]].round(4).to_string())
    return metrics

If the full run has been done, its summary for each scenario is shown below.

In [17]:
for scenario in ("clusters", "no_clusters", "drift"):
    path = RESULTS / scenario / "summary.csv"
    if path.exists():
        s = pd.read_csv(path, index_col="model")
        print(f"\n{scenario}")
        display(s[["crps", "crps_vs_ref", "crps_vs_ref_sd", "folds_better", "crps_cold_start",
                   "brier_late", "coverage_90", "brier_shortage", "precision_at_10"]].round(4))
    else:
        print(f"{scenario}: no full results yet (run src/evaluate_models.py)")

clusters: no full results yet (run src/evaluate_models.py)
no_clusters: no full results yet (run src/evaluate_models.py)
drift: no full results yet (run src/evaluate_models.py)


## Reading the results

The question from the README is whether **clusters improve delay forecasts beyond hierarchical pooling**. The comparisons that answer it:

- **Model 2 vs 1:** do clusters help a model without supplier effects?
- **Model 3 vs 1:** how much does partial pooling help on its own?
- **Model 4 vs 3:** do clusters add anything *once pooling is in place*? This is the key comparison.

Then compare the pattern **across scenarios**, not the raw scores:

- **clusters:** clusters should help model 2. The gap between models 4 and 3 is the real test.
- **no_clusters:** clusters should add little or nothing.
- **drift:** cluster labels go stale. Watch whether model 4 falls behind model 3 here.

Check `crps_cold_start` separately. Hierarchical models earn their keep on suppliers with little history, and the overall average can hide that.